In [1]:
from pathlib import Path
import numpy as np
from astroquery.mast import Observations
from astropy.io import fits
from matplotlib import pyplot as plt

DOWNLOAD_DIR = Path.cwd() / "mast_miri_darks"
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
# MIRI Imager darks are calibration exposures with the OPAQUE filter
# (target_name is usually UNKNOWN, not "DARK").
obs_table = Observations.query_criteria(
    obs_collection="JWST",
    instrument_name="MIRI/IMAGE",
    intentType="calibration",
    filters="OPAQUE",
)

print(f"Found {len(obs_table)} MIRI/IMAGE OPAQUE calibration observations")
obs_table[["obs_id", "instrument_name", "filters", "t_exptime", "proposal_id"]][:10]


Found 439 MIRI/IMAGE OPAQUE calibration observations


obs_id,instrument_name,filters,t_exptime,proposal_id
str34,str10,str6,float64,str5
jw01517021001_02201_00001_mirimage,MIRI/IMAGE,OPAQUE,90.455,1517
jw01517164001_03201_00001_mirimage,MIRI/IMAGE,OPAQUE,1390.295,1517
jw01517174001_02201_00001_mirimage,MIRI/IMAGE,OPAQUE,1390.295,1517
jw01517157001_06201_00001_mirimage,MIRI/IMAGE,OPAQUE,1390.295,1517
jw01517161001_03201_00001_mirimage,MIRI/IMAGE,OPAQUE,1390.295,1517
jw01517041001_02201_00001_mirimage,MIRI/IMAGE,OPAQUE,35.95,1517
jw01517156001_02201_00001_mirimage,MIRI/IMAGE,OPAQUE,1390.295,1517
jw01517019001_03201_00001_mirimage,MIRI/IMAGE,OPAQUE,90.455,1517
jw01517182001_02201_00001_mirimage,MIRI/IMAGE,OPAQUE,1390.295,1517


In [3]:
# Pin the specific MIRI dark used for the gain match below.
TARGET_OBS_ID = "jw01517021001_02201_00001_mirimage"
TARGET_DARK_NAME = f"{TARGET_OBS_ID}_dark.fits"

if len(obs_table) == 0:
    raise RuntimeError("No MIRI/IMAGE OPAQUE calibration observations found.")

match = obs_table[obs_table["obs_id"] == TARGET_OBS_ID]
if len(match) == 0:
    raise RuntimeError(f"{TARGET_OBS_ID} not in MAST query results.")

products = Observations.get_product_list(match[0])
dark_products = Observations.filter_products(
    products,
    productSubGroupDescription=["DARK"],
)
if len(dark_products) == 0:
    raise RuntimeError("No DARK products found for this MIRI observation.")

print(dark_products["productFilename"][:5])
manifest = Observations.download_products(
    dark_products[:1],
    download_dir=str(DOWNLOAD_DIR),
)
print(manifest)

dark_path = Path(manifest["Local Path"][0])
if dark_path.name != TARGET_DARK_NAME:
    raise RuntimeError(f"Expected {TARGET_DARK_NAME}, got {dark_path.name}")
print("Downloaded MIRI dark frame:", dark_path)


              productFilename               
--------------------------------------------
jw01517021001_02201_00001_mirimage_dark.fits
INFO: Found cached file /Users/eckhartspalding/Documents/git.repos/life_detectors/dev_notebooks/mast_miri_darks/mastDownload/JWST/jw01517021001_02201_00001_mirimage/jw01517021001_02201_00001_mirimage_dark.fits with expected size 118304640. [astroquery.query]
                                                                                        Local Path                                                                                        ...
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- ...
/Users/eckhartspalding/Documents/git.repos/life_detectors/dev_notebooks/mast_miri_darks/mastDownload/JWST/jw01517021001_02201_00001_mirimage/jw01517021001_02201_00001_mirimage_dark.fits ...
Downloaded MIRI dark frame: /Users/e

In [4]:
with fits.open(dark_path) as hdul:
    sci = hdul["SCI"].data
    hdr = hdul["SCI"].header
    print(sci.shape)   # expect (5, 250, 128, 136) = (nints, ngroups, ny, nx)
    print(hdr.get("BUNIT"))


(3, 100, 256, 256)
DN


In [ ]:
# Resolve the CRDS GAIN reference that matches this dark's header / context.
# MIRI gain rmaps key on DETECTOR + FILTER (+ BAND, SUBARRAY), not only SUBARRAY.
# Prefer the dark's CRDS_CTX so we get the same file the pipeline context would pick.
import os
import re
import urllib.request

CRDS_PATH = Path.home() / "crds_cache"
CRDS_PATH.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("CRDS_PATH", str(CRDS_PATH))
os.environ.setdefault("CRDS_SERVER_URL", "https://jwst-crds.stsci.edu")

bad_server_config = CRDS_PATH / "config" / "jwst" / "server_config"
if bad_server_config.is_dir():
    import shutil
    shutil.rmtree(bad_server_config)

with fits.open(dark_path) as hdul:
    dark_hdr = hdul[0].header.copy()

gain_header = {
    "INSTRUME": dark_hdr["INSTRUME"],
    "DETECTOR": dark_hdr["DETECTOR"],
    "FILTER": dark_hdr.get("FILTER", "N/A"),
    "SUBARRAY": dark_hdr["SUBARRAY"],
    "DATE-OBS": dark_hdr["DATE-OBS"],
    "TIME-OBS": str(dark_hdr["TIME-OBS"]),
}
crds_ctx = dark_hdr.get("CRDS_CTX")  # e.g. jwst_1535.pmap
print("Dark selection keys:", gain_header)
print("Dark CRDS_CTX:", crds_ctx)


def _parse_pmap_instrument_imap(pmap_path: Path, instrument: str) -> str:
    text = pmap_path.read_text()
    m = re.search(rf"'{instrument}'\s*:\s*'([^']+\.imap)'", text)
    if not m:
        raise RuntimeError(f"No {instrument} imap in {pmap_path.name}")
    return m.group(1)


def _parse_imap_reftype(imap_path: Path, reftype: str) -> str:
    text = imap_path.read_text()
    m = re.search(rf"'{reftype}'\s*:\s*'([^']+\.rmap)'", text)
    if not m:
        raise RuntimeError(f"No {reftype} rmap in {imap_path.name}")
    return m.group(1)


def _resolve_gain_filename_from_rmap(rmap_path: Path, detector: str, filt: str, date_obs: str) -> str:
    """Match MIRIMAGE (+ FILTER) UseAfter entry; SUBARRAY GENERIC covers SUB256."""
    text = rmap_path.read_text()
    # Prefer exact FILTER match; fall back to FILTER N/A.
    # Rmap entries look like: ('MIRIMAGE', 'OPAQUE', 'N/A', 'GENERIC') : UseAfter({...}),
    patterns = [
        rf"\('{detector}', '{filt}', 'N/A', 'GENERIC'\)\s*:\s*UseAfter\(\{{(.*?)\}}\)",
        rf"\('{detector}', 'N/A', 'N/A', 'GENERIC'\)\s*:\s*UseAfter\(\{{(.*?)\}}\)",
    ]
    block = None
    for pat in patterns:
        m = re.search(pat, text, flags=re.S)
        if m:
            block = m.group(1)
            break
    if block is None:
        raise RuntimeError(f"No MIRIMAGE gain UseAfter for FILTER={filt} in {rmap_path.name}")

    # Pick latest USEAFTER date on/before DATE-OBS
    entries = re.findall(r"'(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})'\s*:\s*'([^']+\.fits)'", block)
    if not entries:
        raise RuntimeError(f"No UseAfter dates in matched block of {rmap_path.name}")
    date_key = f"{date_obs} 00:00:00"
    chosen = None
    for useafter, fname in sorted(entries):
        if useafter <= date_key:
            chosen = fname
    if chosen is None:
        chosen = sorted(entries)[0][1]
    return chosen


def resolve_miri_gain_path(header: dict, context: str | None) -> Path:
    """Return local path to the GAIN reference for this observation."""
    # 1) Try official CRDS client when it works in this env.
    try:
        import crds

        refs = crds.getreferences(
            {
                "INSTRUME": header["INSTRUME"],
                "DETECTOR": header["DETECTOR"],
                "FILTER": header["FILTER"],
                "SUBARRAY": header["SUBARRAY"],
                "DATE-OBS": header["DATE-OBS"],
                "TIME-OBS": header["TIME-OBS"],
            },
            reftypes=["gain"],
            observatory="jwst",
            **({"context": context} if context else {}),
        )
        return Path(refs["gain"])
    except Exception as exc:
        print(f"crds.getreferences unavailable ({type(exc).__name__}: {exc})")
        print("Falling back to rmap parse + direct download.")

    # 2) Fallback: walk pmap → imap → gain rmap using the dark's context.
    if not context:
        raise RuntimeError("Need CRDS_CTX on the dark for rmap fallback.")
    maps = CRDS_PATH / "mappings" / "jwst"
    refs_dir = CRDS_PATH / "references" / "jwst" / "miri"
    refs_dir.mkdir(parents=True, exist_ok=True)

    pmap = maps / context
    if not pmap.exists():
        raise FileNotFoundError(f"Missing mapping {pmap}; sync CRDS mappings first.")

    imap_name = _parse_pmap_instrument_imap(pmap, header["INSTRUME"])
    rmap_name = _parse_imap_reftype(maps / imap_name, "GAIN")
    gain_name = _resolve_gain_filename_from_rmap(
        maps / rmap_name,
        detector=header["DETECTOR"],
        filt=header["FILTER"],
        date_obs=header["DATE-OBS"],
    )
    dest = refs_dir / gain_name
    if not dest.exists():
        url = f"https://jwst-crds.stsci.edu/unchecked_get/references/jwst/{gain_name}"
        print("Downloading", url)
        urllib.request.urlretrieve(url, dest)
    return dest


gain_path = resolve_miri_gain_path(gain_header, crds_ctx)
print("CRDS gain file:", gain_path)


In [ ]:
# Load pixel-by-pixel gain and crop to this dark's SUB256 window.
with fits.open(gain_path) as ghdul:
    ghdul.info()
    gain_full = np.asarray(ghdul["SCI"].data, dtype=np.float64)
    gain_bunit = ghdul["SCI"].header.get("BUNIT")
    print(
        "gain ref:",
        f"DETECTOR={ghdul[0].header.get('DETECTOR')}",
        f"FILTER={ghdul[0].header.get('FILTER')}",
        f"SUBARRAY={ghdul[0].header.get('SUBARRAY')}",
        f"BUNIT={gain_bunit}",
        f"shape={gain_full.shape}",
    )

# FITS SUBSTRT* are 1-based; numpy slices are 0-based.
x0 = int(dark_hdr["SUBSTRT1"]) - 1
y0 = int(dark_hdr["SUBSTRT2"]) - 1
nx = int(dark_hdr["SUBSIZE1"])
ny = int(dark_hdr["SUBSIZE2"])
gain = gain_full[y0 : y0 + ny, x0 : x0 + nx]
print(
    f"Cropped gain to SUBARRAY={dark_hdr['SUBARRAY']} "
    f"[{y0}:{y0 + ny}, {x0}:{x0 + nx}] -> {gain.shape}"
)
print(
    "gain e-/DN: "
    f"min={np.nanmin(gain):.4g} max={np.nanmax(gain):.4g} "
    f"median={np.nanmedian(gain):.4g} mean={np.nanmean(gain):.4g}"
)

# Example: convert a DN/s rate map for this dark into e-/pix/s
# rate_e = rate_dn * gain


In [5]:
# convert ramp -> rate

'''

# seconds between groups
tgroup = hdr.get("TGROUP") or hdul[0].header.get("TGROUP")

# Average adjacent-group differences over ints & groups → DN (or e-) per second
# sci: (nints, ngroups, ny, nx)
dcounts = np.diff(sci, axis=1)          # (nints, ngroups-1, ny, nx)
rate_2d = np.nanmedian(dcounts, axis=(0, 1)) / tgroup   # (ny, nx)
'''

'\n\n# seconds between groups\ntgroup = hdr.get("TGROUP") or hdul[0].header.get("TGROUP")\n\n# Average adjacent-group differences over ints & groups → DN (or e-) per second\n# sci: (nints, ngroups, ny, nx)\ndcounts = np.diff(sci, axis=1)          # (nints, ngroups-1, ny, nx)\nrate_2d = np.nanmedian(dcounts, axis=(0, 1)) / tgroup   # (ny, nx)\n'

In [6]:
# Save the computed rate_2d as a FITS file
'''
rate_fits_path = dark_path.parent / (dark_path.stem + "_rate2d.fits")
hdu = fits.PrimaryHDU(rate_2d)
hdu.writeto(rate_fits_path, overwrite=True)
print(f"Saved rate_2d to {rate_fits_path}")
'''

'\nrate_fits_path = dark_path.parent / (dark_path.stem + "_rate2d.fits")\nhdu = fits.PrimaryHDU(rate_2d)\nhdu.writeto(rate_fits_path, overwrite=True)\nprint(f"Saved rate_2d to {rate_fits_path}")\n'

In [7]:
# CRDS master dark -> 2D rate via OLS fit of full ramps
from pathlib import Path
import numpy as np
from astropy.io import fits

CRDS_DARK = Path.home() / "crds_cache/references/jwst/miri/jwst_miri_dark_0113.fits"
# MIRI Imager FULL FASTR1: one frame per group → TGROUP = TFRAME = 2.775 s
# (CRDS dark header has no TGROUP/TFRAME; see JWST MIRI readout docs)
TGROUP_FULL_FASTR1 = 2.775  # seconds

OUT_RATE = CRDS_DARK.with_name(CRDS_DARK.stem + "_rate2d.fits")
CHUNK_ROWS = 64


In [8]:
def ols_rate_chunk(sci_chunk, t_c, denom):
    """OLS slope per pixel for sci_chunk (nints, ngroups, ny, nx).

    Returns mean rate over integrations, shape (ny, nx), in DN/s.
    """
    # center counts along group axis for numerical stability
    yc = sci_chunk - sci_chunk.mean(axis=1, keepdims=True)
    # slope = sum_g (y_c * t_c) / sum(t_c^2)
    rates = np.einsum("igyx,g->iyx", yc, t_c, optimize=True) / denom
    return rates.mean(axis=0)


with fits.open(CRDS_DARK, memmap=True) as hdul:
    sci = hdul["SCI"].data  # (nints, ngroups, ny, nx)
    nints, ngroups, ny, nx = sci.shape
    assert nints == 2 and ngroups == 360

    t = np.arange(ngroups, dtype=np.float64) * TGROUP_FULL_FASTR1
    t_c = t - t.mean()
    denom = float(np.dot(t_c, t_c))

    rate_2d = np.empty((ny, nx), dtype=np.float32)
    for y0 in range(0, ny, CHUNK_ROWS):
        y1 = min(y0 + CHUNK_ROWS, ny)
        chunk = np.asarray(sci[:, :, y0:y1, :], dtype=np.float64)
        rate_2d[y0:y1, :] = ols_rate_chunk(chunk, t_c, denom).astype(np.float32)
        print(f"fitted rows {y0}:{y1}", flush=True)

print(
    "rate_2d DN/s: "
    f"min={rate_2d.min():.4g} max={rate_2d.max():.4g} "
    f"median={np.nanmedian(rate_2d):.4g} mean={np.nanmean(rate_2d):.4g}"
)

hdr = fits.Header()
hdr["BUNIT"] = "DN/s"
hdr["TGROUP"] = (TGROUP_FULL_FASTR1, "seconds; FULL FASTR1")
hdr["NGROUPS"] = ngroups
hdr["NINTS"] = nints
hdr["METHOD"] = "OLS linear fit over all groups; mean over ints"
hdr["SRCFILE"] = CRDS_DARK.name
fits.PrimaryHDU(rate_2d, header=hdr).writeto(OUT_RATE, overwrite=True)
print("Wrote", OUT_RATE)


fitted rows 0:64
fitted rows 64:128
fitted rows 128:192
fitted rows 192:256
fitted rows 256:320
fitted rows 320:384
fitted rows 384:448
fitted rows 448:512
fitted rows 512:576
fitted rows 576:640
fitted rows 640:704
fitted rows 704:768
fitted rows 768:832
fitted rows 832:896
fitted rows 896:960
fitted rows 960:1024
rate_2d DN/s: min=nan max=nan median=0.108 mean=0.1727
Wrote /Users/eckhartspalding/crds_cache/references/jwst/miri/jwst_miri_dark_0113_rate2d.fits
